In [1]:
import pandas as pd
import numpy as np

train_model = pd.read_csv("../data/processed/train_woe.csv")
test_model = pd.read_csv("../data/processed/test_woe.csv")

print(train_model.shape)
print(test_model.shape)

(70000, 9)
(30000, 9)


In [2]:
#Define  X AND Y
woe_features = [
    "Credit_History_Years_WOE",
    "Credit_Utilization_WOE",
    "Debt_to_Income_WOE",
    "Employment_Length_Years_WOE",
    "Monthly_Income_WOE",
    "Number_of_Delinquencies_30_WOE",
    "Number_of_Delinquencies_60_WOE",
    "Number_of_Delinquencies_90_WOE"
]

X_train = train_model[woe_features]
y_train = train_model["Bad"]

X_test = test_model[woe_features]
y_test = test_model["Bad"]

In [3]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in 

In [4]:
train_pd = model.predict_proba(X_train)[:, 1]
test_pd = model.predict_proba(X_test)[:, 1]

In [5]:
coef_table = pd.DataFrame({
    "Feature": woe_features,
    "Coefficient": model.coef_[0]
})

coef_table

,Feature,Coefficient
0,Credit_History_Years_WOE,-0.951241
1,Credit_Utilization_WOE,-0.797219
2,Debt_to_Income_WOE,-0.681773
3,Employment_Length_Years_WOE,-0.919838
4,Monthly_Income_WOE,-0.256850
5,Number_of_Delinquencies_30_WOE,-0.860103
6,Number_of_Delinquencies_60_WOE,-0.833179
7,Number_of_Delinquencies_90_WOE,-0.921746


In [6]:
from sklearn.metrics import roc_auc_score

train_auc = roc_auc_score(y_train, train_pd)
test_auc = roc_auc_score(y_test, test_pd)

print("Train AUC:", train_auc)
print("Test AUC:", test_auc)

Train AUC: 0.5987196907793187
Test AUC: 0.5968950323499084


In [7]:
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_test, test_pd)

ks = max(tpr - fpr)

print("Test KS:", ks)

Test KS: 0.14548036360356503


In [11]:
iv_table = pd.DataFrame({
    "Feature": [
        "Credit_History_Years",
        "Age",
        "Credit_Utilization",
        "Interest_Rate",
        "Debt_to_Income",
        "Employment_Length_Years",
        "Monthly_Income",
        "Recent_Inquiries_6M",
        "Number_of_Open_Loans",
        "Loan_Amount",
        "Number_of_Credit_Lines",
        "Years_at_Address",
        "Total_Debt",
        "Number_of_Dependents",
        "Loan_Tenure_Months"
    ],
    "IV": [
        0.043025,
        0.042711,
        0.032756,
        0.027769,
        0.026131,
        0.017351,
        0.014822,
        0.009027,
        0.008472,
        0.003726,
        0.001616,
        0.001235,
        0.001100,
        0.000830,
        0.000397
    ]
})

iv_table

,Feature,IV
0,Credit_History_Years,0.043025
1,Age,0.042711
2,Credit_Utilization,0.032756
3,Interest_Rate,0.027769
4,Debt_to_Income,0.026131
5,Employment_Length_Years,0.017351
6,Monthly_Income,0.014822
7,Recent_Inquiries_6M,0.009027
8,Number_of_Open_Loans,0.008472
9,Loan_Amount,0.003726


In [23]:
train_base = pd.read_csv("../data/processed/train_woe.csv")
test_base = pd.read_csv("../data/processed/test_woe.csv")

train_extra = pd.read_csv(
    "../data/processed/train_challenger_woe.csv"
)

test_extra = pd.read_csv(
    "../data/processed/test_challenger_woe.csv"
)

In [24]:
challenger_features = [
    "Credit_History_Years_WOE",
    "Credit_Utilization_WOE",
    "Debt_to_Income_WOE",
    "Employment_Length_Years_WOE",
    "Monthly_Income_WOE",
    "Number_of_Delinquencies_30_WOE",
    "Number_of_Delinquencies_60_WOE",
    "Number_of_Delinquencies_90_WOE",
    "Age_WOE",
    "Interest_Rate_WOE"
]

X_train_ch = pd.concat([
    train_base[woe_features],
    train_extra[["Age_WOE", "Interest_Rate_WOE"]]
], axis=1)

X_test_ch = pd.concat([
    test_base[woe_features],
    test_extra[["Age_WOE", "Interest_Rate_WOE"]]
], axis=1)

y_train_ch = train_base["Bad"]
y_test_ch = test_base["Bad"]

In [25]:
print(X_train_ch.shape)
print(X_test_ch.shape)
print(X_train_ch.columns.tolist())

(70000, 10)
(30000, 10)
['Credit_History_Years_WOE', 'Credit_Utilization_WOE', 'Debt_to_Income_WOE', 'Employment_Length_Years_WOE', 'Monthly_Income_WOE', 'Number_of_Delinquencies_30_WOE', 'Number_of_Delinquencies_60_WOE', 'Number_of_Delinquencies_90_WOE', 'Age_WOE', 'Interest_Rate_WOE']


In [26]:

from sklearn.linear_model import LogisticRegression

challenger_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

challenger_model.fit(X_train_ch, y_train_ch)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in 

In [27]:
from sklearn.metrics import roc_auc_score, roc_curve

train_pd_ch = challenger_model.predict_proba(X_train_ch)[:, 1]
test_pd_ch = challenger_model.predict_proba(X_test_ch)[:, 1]

train_auc_ch = roc_auc_score(y_train_ch, train_pd_ch)
test_auc_ch = roc_auc_score(y_test_ch, test_pd_ch)

fpr, tpr, thresholds = roc_curve(y_test_ch, test_pd_ch)

test_ks_ch = max(tpr - fpr)

print("Challenger Train AUC:", train_auc_ch)
print("Challenger Test AUC:", test_auc_ch)
print("Challenger Test KS:", test_ks_ch)

Challenger Train AUC: 0.5993309186935948
Challenger Test AUC: 0.5962313983285958
Challenger Test KS: 0.14130539204521042


In [28]:
champion_results_train = train_model.copy()
champion_results_test = test_model.copy()

champion_results_train["PD"] = train_pd
champion_results_test["PD"] = test_pd

In [29]:
champion_results_train.to_csv(
    "../data/processed/champion_train_pd.csv",
    index=False
)

champion_results_test.to_csv(
    "../data/processed/champion_test_pd.csv",
    index=False
)

In [30]:
import joblib

joblib.dump(
    model,
    "../outputs/models/champion_logistic_model.pkl"
)

['../outputs/models/champion_logistic_model.pkl']

In [31]:
coef_table = pd.DataFrame({
    "Feature": woe_features,
    "Coefficient": model.coef_[0]
})

coef_table.to_csv(
    "../outputs/metrics/champion_coefficients.csv",
    index=False
)

In [32]:
print(champion_results_train[["PD", "Bad"]].head())
print(champion_results_test[["PD", "Bad"]].head())

         PD  Bad
0  0.069466    0
1  0.071593    0
2  0.062271    0
3  0.064605    0
4  0.033871    0
         PD  Bad
0  0.053388    0
1  0.068223    0
2  0.078715    0
3  0.057718    0
4  0.094873    0


In [33]:
# Scorecard scaling

PDO = 20
BASE_SCORE = 600
BASE_ODDS = 50

train_score = BASE_SCORE + PDO * np.log(
    BASE_ODDS * (1 - train_pd) / train_pd
)

test_score = BASE_SCORE + PDO * np.log(
    BASE_ODDS * (1 - test_pd) / test_pd
)

In [35]:
train_score = pd.Series(train_score)
test_score = pd.Series(test_score)

In [36]:
print(train_score.describe())
print(test_score.describe())

count    70000.000000
mean       731.668995
std          7.090703
min        671.019400
25%        727.579133
50%        732.122564
75%        736.624694
max        748.684779
dtype: float64
count    30000.000000
mean       731.629657
std          7.121666
min        678.192119
25%        727.495896
50%        732.108395
75%        736.621821
max        748.483796
dtype: float64


In [37]:
def assign_risk_band(score):
    if score >= 740:
        return "Low Risk"
    elif score >= 730:
        return "Medium Risk"
    elif score >= 720:
        return "High Risk"
    else:
        return "Very High Risk"

In [38]:
train_risk_band = train_score.apply(assign_risk_band)
test_risk_band = test_score.apply(assign_risk_band)

In [39]:
print(train_risk_band.value_counts().sort_index())
print(test_risk_band.value_counts().sort_index())

High Risk         22220
Low Risk           7623
Medium Risk       36250
Very High Risk     3907
Name: count, dtype: int64
High Risk          9605
Low Risk           3279
Medium Risk       15372
Very High Risk     1744
Name: count, dtype: int64


In [40]:
test_model["Risk_Band"] = test_risk_band.values

risk_band_summary = (
    test_model
    .groupby("Risk_Band")
    .agg(
        Customers=("Bad", "count"),
        Bad_Customers=("Bad", "sum"),
        Bad_Rate=("Bad", "mean")
    )
    .reset_index()
)

risk_band_summary["Bad_Rate"] = (
    risk_band_summary["Bad_Rate"] * 100
).round(2)

risk_band_summary

,Risk_Band,Customers,Bad_Customers,Bad_Rate
0,High Risk,9605,784,8.16
1,Low Risk,3279,143,4.36
2,Medium Risk,15372,876,5.70
3,Very High Risk,1744,244,13.99


In [41]:
# =========================
# MODEL VALIDATION
# =========================

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

# Convert PD into predicted class
threshold = 0.50

test_pred = (test_pd >= threshold).astype(int)

# Confusion Matrix
cm = confusion_matrix(y_test, test_pred)

print("Confusion Matrix:")
print(cm)

print("\nPrecision:", round(precision_score(y_test, test_pred), 4))
print("Recall:", round(recall_score(y_test, test_pred), 4))
print("F1 Score:", round(f1_score(y_test, test_pred), 4))

print("\nClassification Report:")
print(classification_report(y_test, test_pred))

Confusion Matrix:
[[27953     0]
 [ 2046     1]]

Precision: 1.0
Recall: 0.0005
F1 Score: 0.001

Classification Report:
              precision    recall  f1-score   support

           0       0.93      1.00      0.96     27953
           1       1.00      0.00      0.00      2047

    accuracy                           0.93     30000
   macro avg       0.97      0.50      0.48     30000
weighted avg       0.94      0.93      0.90     30000



In [ ]:
#or a credit-risk model, don't rely only on the 0.50 threshold. 
# In real credit risk, the threshold is usually selected based 
#on business cost, approval rate, bad rate, and risk appetite.

In [42]:
thresholds = [0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.10]

for threshold in thresholds:
    pred = (test_pd >= threshold).astype(int)

    print(
        "Threshold:", threshold,
        "| Predicted Bad:", pred.sum(),
        "| Recall:", round(recall_score(y_test, pred), 4),
        "| Precision:", round(precision_score(y_test, pred, zero_division=0), 4),
        "| F1:", round(f1_score(y_test, pred, zero_division=0), 4)
    )

Threshold: 0.03 | Predicted Bad: 29985 | Recall: 1.0 | Precision: 0.0683 | F1: 0.1278
Threshold: 0.04 | Predicted Bad: 28290 | Recall: 0.9624 | Precision: 0.0696 | F1: 0.1299
Threshold: 0.05 | Predicted Bad: 23183 | Recall: 0.8539 | Precision: 0.0754 | F1: 0.1386
Threshold: 0.06 | Predicted Bad: 17088 | Recall: 0.6952 | Precision: 0.0833 | F1: 0.1487
Threshold: 0.07 | Predicted Bad: 11305 | Recall: 0.5012 | Precision: 0.0908 | F1: 0.1537
Threshold: 0.08 | Predicted Bad: 7033 | Recall: 0.3483 | Precision: 0.1014 | F1: 0.157
Threshold: 0.1 | Predicted Bad: 2780 | Recall: 0.1646 | Precision: 0.1212 | F1: 0.1396


In [43]:
risk_order = [
    "Low Risk",
    "Medium Risk",
    "High Risk",
    "Very High Risk"
]

risk_band_summary["Risk_Band"] = pd.Categorical(
    risk_band_summary["Risk_Band"],
    categories=risk_order,
    ordered=True
)

risk_band_summary = risk_band_summary.sort_values("Risk_Band")

risk_band_summary

,Risk_Band,Customers,Bad_Customers,Bad_Rate
1,Low Risk,3279,143,4.36
2,Medium Risk,15372,876,5.70
0,High Risk,9605,784,8.16
3,Very High Risk,1744,244,13.99


In [44]:
# Score distribution
train_score_df = pd.DataFrame({"Score": train_score})
test_score_df = pd.DataFrame({"Score": test_score})

print("Train Score Summary:")
print(train_score_df["Score"].describe())

print("\nTest Score Summary:")
print(test_score_df["Score"].describe())

Train Score Summary:
count    70000.000000
mean       731.668995
std          7.090703
min        671.019400
25%        727.579133
50%        732.122564
75%        736.624694
max        748.684779
Name: Score, dtype: float64

Test Score Summary:
count    30000.000000
mean       731.629657
std          7.121666
min        678.192119
25%        727.495896
50%        732.108395
75%        736.621821
max        748.483796
Name: Score, dtype: float64


In [46]:


# Create common bins
bins = np.linspace(
    min(train_score.min(), test_score.min()),
    max(train_score.max(), test_score.max()),
    11
)

# Convert scores into the same 10 bins
train_bins = pd.cut(train_score, bins=bins, include_lowest=True)
test_bins = pd.cut(test_score, bins=bins, include_lowest=True)

# Calculate proportions
train_dist = train_bins.value_counts(sort=False).values / len(train_score)
test_dist = test_bins.value_counts(sort=False).values / len(test_score)

# Replace zero proportions
train_dist = np.where(train_dist == 0, 0.0001, train_dist)
test_dist = np.where(test_dist == 0, 0.0001, test_dist)

# PSI
psi = np.sum(
    (test_dist - train_dist) *
    np.log(test_dist / train_dist)
)

print("PSI:", round(psi, 4))

PSI: 0.0006


In [47]:
from sklearn.metrics import roc_auc_score

# Gini
gini = 2 * roc_auc_score(y_test, test_pd) - 1

# Final validation metrics
validation_summary = pd.DataFrame({
    "Metric": [
        "Test AUC",
        "Gini",
        "KS",
        "PSI"
    ],
    "Value": [
        roc_auc_score(y_test, test_pd),
        gini,
        0.1454803636,
        0.0006
    ]
})

validation_summary["Value"] = validation_summary["Value"].round(4)

validation_summary

,Metric,Value
0,Test AUC,0.5969
1,Gini,0.1938
2,KS,0.1455
3,PSI,0.0006


In [49]:
coef_table


,Feature,Coefficient
0,Credit_History_Years_WOE,-0.951241
1,Credit_Utilization_WOE,-0.797219
2,Debt_to_Income_WOE,-0.681773
3,Employment_Length_Years_WOE,-0.919838
4,Monthly_Income_WOE,-0.256850
5,Number_of_Delinquencies_30_WOE,-0.860103
6,Number_of_Delinquencies_60_WOE,-0.833179
7,Number_of_Delinquencies_90_WOE,-0.921746


In [ ]:
PDO = 20
BASE_SCORE = 600
BASE_ODDS = 50

FACTOR = PDO / np.log(2)
OFFSET = BASE_SCORE - FACTOR * np.log(BASE_ODDS)

coef_table["Factor"] = FACTOR
coef_table["Offset"] = OFFSET

coef_table

,Feature,Coefficient,Factor,Offset
0,Credit_History_Years_WOE,-0.951241,28.853901,487.122876
1,Credit_Utilization_WOE,-0.797219,28.853901,487.122876
2,Debt_to_Income_WOE,-0.681773,28.853901,487.122876
3,Employment_Length_Years_WOE,-0.919838,28.853901,487.122876
4,Monthly_Income_WOE,-0.256850,28.853901,487.122876
5,Number_of_Delinquencies_30_WOE,-0.860103,28.853901,487.122876
6,Number_of_Delinquencies_60_WOE,-0.833179,28.853901,487.122876
7,Number_of_Delinquencies_90_WOE,-0.921746,28.853901,487.122876


In [51]:
coef_table["Points_per_WOE"] = -coef_table["Coefficient"] * FACTOR

coef_table[
    ["Feature", "Coefficient", "Points_per_WOE"]
]

,Feature,Coefficient,Points_per_WOE
0,Credit_History_Years_WOE,-0.951241,27.447012
1,Credit_Utilization_WOE,-0.797219,23.002874
2,Debt_to_Income_WOE,-0.681773,19.671813
3,Employment_Length_Years_WOE,-0.919838,26.540914
4,Monthly_Income_WOE,-0.256850,7.411115
5,Number_of_Delinquencies_30_WOE,-0.860103,24.817336
6,Number_of_Delinquencies_60_WOE,-0.833179,24.040463
7,Number_of_Delinquencies_90_WOE,-0.921746,26.595964


In [54]:
import pandas as pd
import numpy as np

def fit_woe_transformer(df, feature, target="Bad", n_bins=10):

    temp = df[[feature, target]].copy()

    temp["Bin"] = temp[feature].astype(object)
    temp.loc[temp[feature].isna(), "Bin"] = "Missing"

    n_unique = temp[feature].nunique()

    if n_unique <= 5:

        temp.loc[temp[feature].notna(), "Bin"] = (
            temp.loc[temp[feature].notna(), feature].astype(str)
        )

        bin_edges = None

    else:

        non_missing = temp[feature].notna()

        temp.loc[non_missing, "Bin"], bin_edges = pd.qcut(
            temp.loc[non_missing, feature],
            q=n_bins,
            retbins=True,
            duplicates="drop"
        )

        bin_edges[0] = -np.inf
        bin_edges[-1] = np.inf

    total_good = (temp[target] == 0).sum()
    total_bad = (temp[target] == 1).sum()

    grouped = (
        temp.groupby(
            "Bin",
            dropna=False,
            observed=False
        )[target]
        .agg(total="count", bad="sum")
    )

    grouped["good"] = grouped["total"] - grouped["bad"]

    grouped["dist_good"] = grouped["good"] / total_good
    grouped["dist_bad"] = grouped["bad"] / total_bad

    grouped["dist_good"] = grouped["dist_good"].replace(0, 0.000001)
    grouped["dist_bad"] = grouped["dist_bad"].replace(0, 0.000001)

    grouped["WOE"] = np.log(
        grouped["dist_good"] / grouped["dist_bad"]
    )

    grouped["IV"] = (
        grouped["dist_good"] - grouped["dist_bad"]
    ) * grouped["WOE"]

    return {
        "bin_edges": bin_edges,
        "woe_table": grouped
    }

print("Function created successfully.")

Function created successfully.


In [55]:
train_raw = pd.read_csv("../data/processed/train_data.csv")

woe_transformers = {}

features_for_scorecard = [
    "Credit_History_Years",
    "Credit_Utilization",
    "Debt_to_Income",
    "Employment_Length_Years",
    "Monthly_Income",
    "Number_of_Delinquencies_30",
    "Number_of_Delinquencies_60",
    "Number_of_Delinquencies_90"
]

for feature in features_for_scorecard:

    woe_transformers[feature] = fit_woe_transformer(
        train_raw,
        feature,
        target="Bad",
        n_bins=10
    )

print("Created:", len(woe_transformers), "WOE transformers")

Created: 8 WOE transformers


In [56]:
scorecard_rows = []

for feature in woe_features:

    base_feature = feature.replace("_WOE", "")

    transformer = woe_transformers[base_feature]
    woe_table = transformer["woe_table"]

    coefficient = model.coef_[0][
        woe_features.index(feature)
    ]

    points_per_woe = -coefficient * FACTOR

    for bin_value, row in woe_table.iterrows():

        points = row["WOE"] * points_per_woe

        scorecard_rows.append({
            "Feature": base_feature,
            "Bin": str(bin_value),
            "WOE": row["WOE"],
            "Coefficient": coefficient,
            "Points": points
        })

scorecard = pd.DataFrame(scorecard_rows)

scorecard["Points"] = scorecard["Points"].round(2)

scorecard

,Feature,Bin,WOE,Coefficient,Points
0,Credit_History_Years,"(0.499, 5.9]",-0.335751,-0.951241,-9.22
1,Credit_History_Years,"(5.9, 10.9]",-0.197876,-0.951241,-5.43
2,Credit_History_Years,"(10.9, 14.4]",-0.141653,-0.951241,-3.89
3,Credit_History_Years,"(14.4, 17.5]",-0.088343,-0.951241,-2.42
4,Credit_History_Years,"(17.5, 20.4]",-0.006497,-0.951241,-0.18
...,...,...,...,...,...
58,Number_of_Delinquencies_60,2,-1.072406,-0.833179,-25.78
59,Number_of_Delinquencies_60,3,-1.390449,-0.833179,-33.43
60,Number_of_Delinquencies_90,0,0.010241,-0.921746,0.27
61,Number_of_Delinquencies_90,1,-0.850636,-0.921746,-22.62


In [57]:
scorecard.to_csv(
    "../outputs/scorecard/final_scorecard.csv",
    index=False
)

print("Final scorecard saved successfully.")

Final scorecard saved successfully.


In [58]:
print(scorecard.head(10))

                Feature           Bin       WOE  Coefficient  Points
0  Credit_History_Years  (0.499, 5.9] -0.335751    -0.951241   -9.22
1  Credit_History_Years   (5.9, 10.9] -0.197876    -0.951241   -5.43
2  Credit_History_Years  (10.9, 14.4] -0.141653    -0.951241   -3.89
3  Credit_History_Years  (14.4, 17.5] -0.088343    -0.951241   -2.42
4  Credit_History_Years  (17.5, 20.4] -0.006497    -0.951241   -0.18
5  Credit_History_Years  (20.4, 23.2]  0.056292    -0.951241    1.55
6  Credit_History_Years  (23.2, 26.3]  0.045535    -0.951241    1.25
7  Credit_History_Years  (26.3, 29.9]  0.254713    -0.951241    6.99
8  Credit_History_Years  (29.9, 34.9]  0.284889    -0.951241    7.82
9  Credit_History_Years  (34.9, 40.0]  0.325807    -0.951241    8.94


In [59]:
print("Model score mean:", round(test_score.mean(), 2))
print("Model score min :", round(test_score.min(), 2))
print("Model score max :", round(test_score.max(), 2))

Model score mean: 731.63
Model score min : 678.19
Model score max : 748.48


In [60]:
print("Scorecard rows:", len(scorecard))
print("Expected rows: 63")

Scorecard rows: 63
Expected rows: 63


In [62]:
test_woe = pd.read_csv("../data/processed/test_woe.csv")

print(test_woe.shape)
print(test_woe[woe_features].isna().sum().sum())

(30000, 9)
0


In [63]:
# Score reconstructed from WOE values
test_score_reconstructed = (
    OFFSET +
    sum(
        test_woe[feature] *
        (-model.coef_[0][i] * FACTOR)
        for i, feature in enumerate(woe_features)
    )
)

# Compare with original model score
difference = test_score_reconstructed - test_score

print("Maximum absolute difference:",
      round(np.abs(difference).max(), 6))

print("Mean absolute difference:",
      round(np.abs(difference).mean(), 6))

Maximum absolute difference: 266.579718
Mean absolute difference: 242.923184


In [64]:
FACTOR = PDO / np.log(2)

BASE_SCORE = 600
BASE_ODDS = 50

intercept = model.intercept_[0]

BASE_SCORE_CORRECTED = (
    BASE_SCORE
    + FACTOR * np.log(BASE_ODDS)
    - FACTOR * intercept
)

print("Model intercept:", round(intercept, 6))
print("Corrected base score:", round(BASE_SCORE_CORRECTED, 2))

Model intercept: -2.614577
Corrected base score: 788.32


In [65]:
test_score_reconstructed = (
    BASE_SCORE_CORRECTED +
    sum(
        test_woe[feature] *
        (-model.coef_[0][i] * FACTOR)
        for i, feature in enumerate(woe_features)
    )
)

difference = test_score_reconstructed - test_score

print(
    "Maximum absolute difference:",
    round(np.abs(difference).max(), 6)
)

print(
    "Mean absolute difference:",
    round(np.abs(difference).mean(), 6)
)

Maximum absolute difference: 65.73304
Mean absolute difference: 58.271797


In [66]:
print("Intercept:", model.intercept_[0])

coef_check = pd.DataFrame({
    "Feature": woe_features,
    "Coefficient": model.coef_[0]
})

coef_check

Intercept: -2.614576575618697


,Feature,Coefficient
0,Credit_History_Years_WOE,-0.951241
1,Credit_Utilization_WOE,-0.797219
2,Debt_to_Income_WOE,-0.681773
3,Employment_Length_Years_WOE,-0.919838
4,Monthly_Income_WOE,-0.256850
5,Number_of_Delinquencies_30_WOE,-0.860103
6,Number_of_Delinquencies_60_WOE,-0.833179
7,Number_of_Delinquencies_90_WOE,-0.921746


In [67]:
print("Original score mean:", round(test_score.mean(), 2))
print("Original score min :", round(test_score.min(), 2))
print("Original score max :", round(test_score.max(), 2))

print("\nStandard scaling:")
print("Factor:", round(FACTOR, 6))
print("Offset:", round(OFFSET, 6))

Original score mean: 731.63
Original score min : 678.19
Original score max : 748.48

Standard scaling:
Factor: 28.853901
Offset: 487.122876


In [68]:
test_score_standard = (
    OFFSET
    - FACTOR * (
        model.intercept_[0]
        + sum(
            test_woe[feature] * model.coef_[0][i]
            for i, feature in enumerate(woe_features)
        )
    )
)

print("Standard score mean:",
      round(test_score_standard.mean(), 2))

print("Standard score min:",
      round(test_score_standard.min(), 2))

print("Standard score max:",
      round(test_score_standard.max(), 2))

Standard score mean: 564.15
Standard score min: 487.05
Standard score max: 588.46


In [69]:
test_score_standard

0        570.086837
1        562.556034
2        558.101619
3        567.704276
4        552.203731
            ...    
29995    555.965872
29996    569.530914
29997    572.368817
29998    564.833354
29999    558.613847
Length: 30000, dtype: float64

In [70]:
test_score = test_score_standard

test_model["Score"] = test_score

test_model.to_csv(
    "../data/processed/champion_test_pd.csv",
    index=False
)

print("Final standard score saved.")
print("Mean:", round(test_score.mean(), 2))
print("Min :", round(test_score.min(), 2))
print("Max :", round(test_score.max(), 2))

Final standard score saved.
Mean: 564.15
Min : 487.05
Max : 588.46


In [71]:
def assign_risk_band(score):
    if score >= 575:
        return "Low Risk"
    elif score >= 565:
        return "Medium Risk"
    elif score >= 550:
        return "High Risk"
    else:
        return "Very High Risk"


test_risk_band = test_score.apply(assign_risk_band)

test_model["Risk_Band"] = test_risk_band

print(test_model["Risk_Band"].value_counts())

Risk_Band
High Risk         12629
Medium Risk       10627
Low Risk           4183
Very High Risk     2561
Name: count, dtype: int64


In [72]:
risk_band_summary = (
    test_model
    .groupby("Risk_Band", observed=False)
    .agg(
        Customers=("Bad", "count"),
        Bad_Customers=("Bad", "sum"),
        Bad_Rate=("Bad", "mean")
    )
    .reset_index()
)

risk_band_summary["Bad_Rate"] = (
    risk_band_summary["Bad_Rate"] * 100
).round(2)

risk_band_summary

,Risk_Band,Customers,Bad_Customers,Bad_Rate
0,High Risk,12629,984,7.79
1,Low Risk,4183,174,4.16
2,Medium Risk,10627,573,5.39
3,Very High Risk,2561,316,12.34


In [73]:
# Standard scorecard points
scorecard["Points"] = (
    scorecard["WOE"]
    * scorecard["Coefficient"]
    * (-FACTOR)
).round(2)

scorecard.head(10)

,Feature,Bin,WOE,Coefficient,Points
0,Credit_History_Years,"(0.499, 5.9]",-0.335751,-0.951241,-9.22
1,Credit_History_Years,"(5.9, 10.9]",-0.197876,-0.951241,-5.43
2,Credit_History_Years,"(10.9, 14.4]",-0.141653,-0.951241,-3.89
3,Credit_History_Years,"(14.4, 17.5]",-0.088343,-0.951241,-2.42
4,Credit_History_Years,"(17.5, 20.4]",-0.006497,-0.951241,-0.18
5,Credit_History_Years,"(20.4, 23.2]",0.056292,-0.951241,1.55
6,Credit_History_Years,"(23.2, 26.3]",0.045535,-0.951241,1.25
7,Credit_History_Years,"(26.3, 29.9]",0.254713,-0.951241,6.99
8,Credit_History_Years,"(29.9, 34.9]",0.284889,-0.951241,7.82
9,Credit_History_Years,"(34.9, 40.0]",0.325807,-0.951241,8.94


In [74]:
scorecard.to_csv(
    "../outputs/scorecard/final_scorecard.csv",
    index=False
)

print("Standard-scaled scorecard saved successfully.")

Standard-scaled scorecard saved successfully.


In [75]:
print(scorecard.shape)

(63, 5)


In [76]:
test_score_from_model = (
    OFFSET
    - FACTOR * (
        model.intercept_[0]
        + test_woe[woe_features].values @ model.coef_[0]
    )
)

difference = test_score_standard - test_score_from_model

print("Maximum absolute difference:",
      round(np.abs(difference).max(), 6))

print("Mean absolute difference:",
      round(np.abs(difference).mean(), 6))

Maximum absolute difference: 0.0
Mean absolute difference: 0.0


In [77]:
validation_summary.to_csv(
    "../outputs/metrics/final_validation_metrics.csv",
    index=False
)

print("Validation metrics saved.")


Validation metrics saved.


In [ ]:
import joblib

woe_transformers = {}

for feature in final_features:
    woe_transformers[feature] = fit_woe_transformer(
        train_df,
        feature,
        target="Bad",
        n_bins=10
    )

joblib.dump(
    woe_transformers,
    "../outputs/models/woe_transformers.pkl"
)

print("WOE transformers saved successfully.")

In [1]:
import joblib

woe_transformers = {}

for feature in final_features:
    woe_transformers[feature] = fit_woe_transformer(
        train_df,
        feature,
        target="Bad",
        n_bins=10
    )

joblib.dump(
    woe_transformers,
    "../outputs/models/woe_transformers.pkl"
)

print("WOE transformers saved successfully.")

NameError: name 'final_features' is not defined